# Detecção de Anomalias em Transações em Python
### Bootcamp Bradesco DIO - Projeto

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

# Dataset: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
df = pd.read_csv('creditcard.csv')
print(df['Class'].value_counts())


In [ ]:
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
X = df.drop(['Class','Amount','Time'], axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [ ]:
# 1. Baseline - Isolation Forest
iso = IsolationForest(contamination=0.0017, random_state=42)
iso.fit(X_train)
y_pred_iso = np.where(iso.predict(X_test)==-1, 1, 0)
print(classification_report(y_test, y_pred_iso))


In [ ]:
# 2. Balanceamento SMOTE + Random Forest
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_train, y_train)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_sm, y_sm)
print(classification_report(y_test, rf.predict(X_test)))


In [ ]:
# 3. XGBoost + SHAP - Modelos Avançados e Explicabilidade
model = xgb.XGBClassifier(scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]), eval_metric='logloss')
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, max_display=10)
